In [0]:
spark.sql("USE CATALOG iitb")
spark.sql("USE SCHEMA bharat_bricks")

In [0]:
#Path=/Volumes/iitb/bharat_bricks/data/stations.json

In [0]:
posts_path = "/Volumes/iitb/bharat_bricks/data/stations.json"

posts_df = (spark.read
            .format("json")
            .option("multiLine", "true")
            .load(posts_path))

posts_df.write.format("delta").mode("overwrite").saveAsTable("Stations")

In [0]:
from pyspark.sql.functions import explode, col

# Explode the features array to show individual station records
stations_exploded = spark.table("Stations").select(explode(col("features")).alias("feature"))

print(f"Total stations: {stations_exploded.count()}")
print("\nShowing top 10 stations:")
display(stations_exploded.limit(10))

In [0]:
stations_flat = stations_exploded.select(
    col("feature.properties.code").alias("code"),
    col("feature.properties.name").alias("name"),
    col("feature.properties.state").alias("state"),
    col("feature.properties.zone").alias("zone"),
    col("feature.geometry.coordinates")[0].alias("longitude"),
    col("feature.geometry.coordinates")[1].alias("latitude")
)

display(stations_flat.limit(10))

In [0]:
posts_path = "/Volumes/iitb/bharat_bricks/data/schedules.json"

posts_df = (spark.read
            .format("json")
            .option("multiLine", "true")
            .load(posts_path))

posts_df.write.format("delta").mode("overwrite").saveAsTable("Schedules")

In [0]:
display(spark.sql("SELECT * FROM Schedules LIMIT 10"))

In [0]:
posts_path = "/Volumes/iitb/bharat_bricks/data/trains.json"

posts_df = (spark.read
            .format("json")
            .option("multiLine", "true")
            .load(posts_path))

posts_df.write.format("delta").mode("overwrite").saveAsTable("trains")

In [0]:
from pyspark.sql.functions import explode, col

# Explode the features array to show individual train records
trains_exploded = spark.table("trains").select(explode(col("features")).alias("feature"))

print(f"Total trains: {trains_exploded.count()}")
print("\nShowing top 10 trains:")
display(trains_exploded.limit(10))

In [0]:
from pyspark.sql.functions import explode, col

trains_exploded = spark.table("trains") \
    .select(explode(col("features")).alias("feature"))


In [0]:
trains_flat = trains_exploded.select(
    col("feature.properties.number").alias("train_number"),
    col("feature.properties.name").alias("train_name"),
    col("feature.properties.from_station_code").alias("from_code"),
    col("feature.properties.from_station_name").alias("from_name"),
    col("feature.properties.to_station_code").alias("to_code"),
    col("feature.properties.to_station_name").alias("to_name"),
    col("feature.properties.distance").alias("distance"),
    col("feature.properties.duration_h").alias("duration_h"),
    col("feature.properties.duration_m").alias("duration_m"),
    col("feature.properties.type").alias("train_type"),
    col("feature.properties.zone").alias("zone"),
    col("feature.geometry.coordinates").alias("route_coordinates")  # IMPORTANT
)

display(trains_flat.limit(10))

DELAY DATASET

In [0]:
posts_path = "/Volumes/iitb/bharat_bricks/data/etrain_delays.csv"

posts_df = (spark.read
            .format("csv")
            .option("header", "true")
            .load(posts_path))

posts_df.write.format("delta").mode("overwrite").saveAsTable("Delays")

In [0]:
display(spark.sql("SELECT * FROM Delays LIMIT 10"))

In [0]:
spark.table("Schedules").printSchema()
spark.table("trains").printSchema()

In [0]:
schedule_trains = spark.table("Schedule").select(
    col("number").cast("string").alias("train_number")
).distinct()

delay_trains = spark.table("Delays").select(
    col("train_no").cast("string").alias("train_number")
).distinct()

In [0]:
common_trains = schedule_trains.join(
    delay_trains,
    on="train_number",
    how="inner"
)
print("Common train numbers:", common_trains.count())

In [0]:
display(common_trains)